# ResNet

In [7]:
from datasets import load_from_disk
loaded_dataset = load_from_disk('./dataset')

print(loaded_dataset)


DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 120004
    })
})


In [8]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [9]:
# Define a transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match CNN input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3-channel grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Custom Dataset class
class DeepFakeDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
        self.start_idx = 4  # Ignore the first 4 rows

    def __len__(self):
        return len(self.dataset['train']) - self.start_idx  # Adjust length

    def __getitem__(self, idx):
        idx += self.start_idx  # Shift index to ignore first 4 rows
        image = self.dataset['train'][idx]['image']  # Assuming it's already a PIL.Image
        label = 0 if idx < 90000 else 1  # First 90k are fake, last 30k are real

        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)


# Create dataset & DataLoader
full_dataset = DeepFakeDataset(loaded_dataset, transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [10]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_curve
import numpy as np

def compute_eer(y_true, y_scores):
    """
    Calculate Equal Error Rate (EER)
    :param y_true: True labels (1 for real, 0 for fake)
    :param y_scores: Model scores (probabilities or decision values)
    :return: EER value, threshold where EER occurs
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr  # False Negative Rate (FRR)
    
    # Find the threshold where FPR ≈ FNR
    eer_threshold = thresholds[np.nanargmin(np.abs(fpr - fnr))]
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    
    return eer, eer_threshold

In [12]:


# Load a pre-trained ResNet model
model = models.resnet18(pretrained=True)

# Modify the classifier layer for binary classification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification (fake vs real)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# K-Fold Cross Validation
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Training loop with K-Fold
num_epochs = 5
all_accuracies = []
all_aucs = []
all_eers = []

for fold, (train_idx, test_idx) in enumerate(kf.split(full_dataset)):
    print(f'Fold {fold+1}/{k_folds}')
    
    train_subset = torch.utils.data.Subset(full_dataset, train_idx.tolist())
    test_subset = torch.utils.data.Subset(full_dataset, test_idx.tolist())
    
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)[:, 1]  
            all_preds.extend(probabilities.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ---- Compute Metrics ----
    accuracy = accuracy_score(all_labels, all_preds > 0.5)
    auc = roc_auc_score(all_labels, all_preds)
    eer_value, eer_threshold = compute_eer(all_labels, all_preds)

    all_accuracies.append(accuracy)
    all_aucs.append(auc)
    all_eers.append(eer_value)

    print(f"Fold {fold+1} - Accuracy: {accuracy:.4f}, AUC: {auc:.4f}, EER: {eer_value:.4f} (Threshold: {eer_threshold:.4f})")

print("Training complete.")
print(f"Average Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Average AUC: {np.mean(all_aucs):.4f}")
print(f"Average EER: {np.mean(all_eers):.4f}")


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Fold 1/5


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Epoch [1/5], Loss: 0.3811
Epoch [2/5], Loss: 0.2738
Epoch [3/5], Loss: 0.2160
Epoch [4/5], Loss: 0.1718
Epoch [5/5], Loss: 0.1368
Fold 1 - Accuracy: 0.8874, AUC: 0.9458, EER: 0.1354 (Threshold: 0.1818)
Fold 2/5


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


...

Fold 3 - Accuracy: 0.8764, AUC: 0.9785, EER: 0.0861 (Threshold: 0.8051)
Fold 4/5


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Epoch [1/5], Loss: 0.0640
Epoch [2/5], Loss: 0.0515
Epoch [3/5], Loss: 0.0453
Epoch [4/5], Loss: 0.0424
Epoch [5/5], Loss: 0.0405
Fold 4 - Accuracy: 0.9515, AUC: 0.9939, EER: 0.0408 (Threshold: 0.0222)
Fold 5/5


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Epoch [1/5], Loss: 0.0472
Epoch [2/5], Loss: 0.0393
Epoch [3/5], Loss: 0.0358
Epoch [4/5], Loss: 0.0346
Epoch [5/5], Loss: 0.0335
Fold 5 - Accuracy: 0.9634, AUC: 0.9944, EER: 0.0398 (Threshold: 0.3450)
Training complete.
Average Accuracy: 0.9171
Average AUC: 0.9771
Average EER: 0.0790
